In [ ]:
import pandas as pd
import plotly.express as px
from typing import Literal

# section,subsection,paragraph,section_text,subsection_text,raw_knowledge_statement,knowledge_statement,atomic_knowledge_statement,atomic_target_span,atomic_knowledge_probe

def plot_interactive_probes(
    metrics_csv_path: str,
    probes_csv_path: str,
    metric_column: str,
    probe_type: Literal['raw', 'atomic'],
    probe_range: tuple = (0, 20)
):
    """
    Creates an interactive plot to visualize the performance of individual knowledge probes
    from the consolidated metrics file.

    Args:
        metrics_csv_path (str):
            Path to the consolidated CSV file containing all probe metrics over time,
            which should now be 'knowledge_probe_metrics.csv'.
        probes_csv_path (str):
            Path to the original CSV file containing the text of the knowledge probes
            (e.g., 'DPO_knowledge_probes.csv').
        metric_column (str):
            The name of the column from the metrics_csv to plot on the y-axis.
            Available columns include:
            - 'raw_knowledge_perplexity'
            - 'raw_knowledge_perplexity_delta'
            - 'atomic_whole_perplexity'
            - 'atomic_whole_perplexity_delta'
            - 'atomic_target_perplexity'
            - 'atomic_target_perplexity_delta'
            - 'atomic_whole_log_prob'
            - 'atomic_whole_log_prob_delta'
            - 'atomic_target_log_prob'
            - 'atomic_target_log_prob_delta'
        probe_type (Literal['raw', 'atomic']):
            Specifies the type of probe to determine how hover text is displayed.
            - 'raw': For raw knowledge statements.
            - 'atomic': For atomic statements (context + target).
        probe_range (tuple, optional):
            A tuple (start, end) specifying the slice of probes to plot by their index.
            Defaults to (0, 20).
    """
    try:
        metrics_df = pd.read_csv(metrics_csv_path)
        probes_df = pd.read_csv(probes_csv_path)
    except FileNotFoundError as e:
        print(f"Error loading files: {e}")
        return

    # Prepare probe text for hovering
    probes_df['probe_index'] = probes_df.index
    if probe_type == 'raw':
        probes_df['display_text'] = "PROBE: " + probes_df['raw_knowledge_statement']
    elif probe_type == 'atomic':
        # Bold the target in the hover text if the metric is target-specific
        if 'target' in metric_column:
            probes_df['display_text'] = "CONTEXT: " + probes_df['atomic_knowledge_probe'] + "<br><b>TARGET: " + probes_df['atomic_target_span'] + "</b>"
        else: # For 'whole' metrics
            probes_df['display_text'] = "CONTEXT: " + probes_df['atomic_knowledge_probe'] + "<br>TARGET: " + probes_df['atomic_target_span']
    else:
        raise ValueError("probe_type must be one of 'raw' or 'atomic'")

    # Merge metric data with probe text. Merge `section` in as well.
    merged_df = pd.merge(metrics_df, probes_df[['probe_index', 'section', 'display_text']], on='probe_index', how='left')

    # Filter for the selected range of probes
    start, end = probe_range
    filtered_df = merged_df[(merged_df['probe_index'] >= start) & (merged_df['probe_index'] < end)].copy()

    if filtered_df.empty:
        print(f"No data found for probe indices in range {probe_range}. Please check the CSV files and range.")
        return
    if metric_column not in filtered_df.columns:
        print(f"Metric column '{metric_column}' not found in '{metrics_csv_path}'.")
        print(f"\nAvailable columns are:\n" + "\n".join(list(filtered_df.columns)))
        return

    # Drop rows where the metric to be plotted is NaN (e.g. raw metrics for atomic probes)
    filtered_df.dropna(subset=[metric_column], inplace=True)
    if filtered_df.empty:
        print(f"All values for metric '{metric_column}' in probe range {probe_range} are NaN.")
        return

    # Create the interactive plot
    title = f'Disaggregated Probes {start}-{end-1} for: {metric_column}'
    fig = px.line(
        filtered_df,
        x='step',
        y=metric_column,
        color='probe_index',
        hover_name='probe_index',
        hover_data=['display_text', 'section'],
        title=title,
        labels={'step': 'Training Step', metric_column: metric_column.replace('_', ' ').title()}
    )
    
    # Customize hover template for better readability
    fig.update_traces(hovertemplate=(
        "<b>Probe %{hovertext}</b><br><br>" +
        "Step: %{x}<br>" +
        f"{metric_column}: %{{y:.3f}}<br>" +
        "Section: %{customdata[1]}<br>" +
        "<hr>" +
        "%{customdata[0]}" + # display_text
        "<extra></extra>" # Hides the trace name
    ))
    
    fig.update_layout(legend_title_text='Probe Index')
    fig.show()

# --- USAGE EXAMPLE ---
#
# Configure the paths and parameters below to match your experiment.

# Path to the directory where the results for a specific experiment were saved.
RESULTS_DIR = '../../results/FT/SingleArxivPaper_1B/'

# Path to the original CSV file with probe definitions.
PROBES_CSV = '../data/arxiv/DPO_knowledge_probes.csv'

# Path to the single, consolidated metrics file.
METRICS_CSV = f'{RESULTS_DIR}/knowledge_probe_metrics.csv'


# --- Example 1: Analyze perplexity delta for "raw knowledge" probes ---
print("--- Plotting Raw Knowledge Probes (Perplexity Delta) ---")
plot_interactive_probes(
    metrics_csv_path=METRICS_CSV,
    probes_csv_path=PROBES_CSV,
    metric_column='raw_knowledge_perplexity_delta',
    probe_type='raw_knowledge_statement',
    probe_range=(0, 25) # Show the first 25 probes
)

# --- Example 2: Analyze log-probability delta for "atomic knowledge target" ---
print("\n--- Plotting Atomic Knowledge Probes (Target Log-Prob Delta) ---")
plot_interactive_probes(
    metrics_csv_path=METRICS_CSV,
    probes_csv_path=PROBES_CSV,
    metric_column='atomic_whole_perplexity_delta',
    probe_type='atomic_knowledge_statement',
    probe_range=(0, 25)
)

# --- Example 3: Analyze perplexity for "atomic knowledge statement" ---
print("\n--- Plotting Atomic Knowledge Statements (Whole Perplexity) ---")
plot_interactive_probes(
    metrics_csv_path=METRICS_CSV,
    probes_csv_path=PROBES_CSV,
    metric_column='atomic_target_perplexity_delta', # Plotting raw perplexity, not the delta
    probe_type='atomic_target_span',
    probe_range=(25, 50) # Show the next 25 probes
)